# RAG Pipeline: Retrieval-Augmented Generation

Build RAG system combining hybrid retrieval with LLM for question answering. Auto-generates supporting modules for document chunking, prompts, and pipeline orchestration.

## Output Files
- **src/chunking.py** - DocumentChunker class (auto-generated)
- **src/prompts.py** - RAGPrompts class with templates (auto-generated)
- **src/rag_pipeline.py** - RAGPipeline class with hybrid retrieval (auto-generated)

## Workflow
1. Setup project paths
2. Create directories
3. Auto-generate src/chunking.py for document chunking
4. Auto-generate src/prompts.py with prompt templates
5. Auto-generate src/rag_pipeline.py with RAG orchestration
6. Verify files generated (lightweight - no heavy imports)

## 1. Setup Project Paths

Detects if running from notebooks/ folder and sets project_root correctly. Must run BEFORE any imports from src.

**Output:** Project root path confirmation

In [1]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
    print("Detected: Running from notebooks/ folder")
else:
    project_root = notebook_dir
    print("Detected: Running from project root")

sys.path.insert(0, str(project_root))

print(f"\nProject root: {project_root}")
print(f"Sys path updated")
print("\nReady for file generation")

Detected: Running from notebooks/ folder

Project root: /Users/esteki/Desktop/MDS/575/DSCI_575_project_jchuang_esteki
Sys path updated

Ready for file generation


## 2. Create Directories

Creates src/ and data/processed directories if they don't exist.

**Output:** Directory confirmation

In [2]:
src_dir = project_root / "src"
data_dir = project_root / "data" / "processed"

src_dir.mkdir(exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

print(f"Directories ready:")
print(f"  src/: {src_dir.exists()}")
print(f"  data/processed/: {data_dir.exists()}")

Directories ready:
  src/: True
  data/processed/: True


## 3. Auto-Generate src/chunking.py

Creates DocumentChunker class for splitting long documents into overlapping chunks.

**Output:** src/chunking.py file created with DocumentChunker class

In [3]:
chunking_code = '''from typing import List


class DocumentChunker:
    """Chunk documents into manageable pieces."""
    
    def __init__(self, chunk_size: int = 500, overlap: int = 50):
        """Initialize chunker with size and overlap parameters."""
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_document(self, text: str) -> List[str]:
        """Split document into chunks with overlap."""
        if len(text) <= self.chunk_size:
            return [text]
        
        chunks = []
        start = 0
        
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            chunks.append(text[start:end])
            start = end - self.overlap
        
        return chunks
'''

chunking_path = project_root / "src" / "chunking.py"
with open(chunking_path, 'w') as f:
    f.write(chunking_code)

print(f"Created: src/chunking.py")
print(f"  Class: DocumentChunker")

Created: src/chunking.py
  Class: DocumentChunker


## 4. Auto-Generate src/prompts.py

Creates RAGPrompts class with configurable prompt templates for different answer styles.

**Output:** src/prompts.py file created with RAGPrompts class

In [4]:
prompts_code = '''class RAGPrompts:
    """RAG prompt templates for different answer styles."""
    
    BALANCED = """Based on the following book reviews and information, answer the question.

Context:
{context}

Question: {question}

Answer:"""
    
    STRICT = """Using ONLY the provided book reviews and information below, answer the question.
If the information is insufficient to answer, say so.

Context:
{context}

Question: {question}

Answer:"""
    
    @staticmethod
    def get_template(version: str = "balanced") -> str:
        """Get prompt template by version name."""
        templates = {
            "balanced": RAGPrompts.BALANCED,
            "strict": RAGPrompts.STRICT
        }
        return templates.get(version.lower(), RAGPrompts.BALANCED)
'''

prompts_path = project_root / "src" / "prompts.py"
with open(prompts_path, 'w') as f:
    f.write(prompts_code)

print(f"Created: src/prompts.py")
print(f"  Class: RAGPrompts")
print(f"  Templates: balanced, strict")

Created: src/prompts.py
  Class: RAGPrompts
  Templates: balanced, strict


## 5. Auto-Generate src/rag_pipeline.py

Creates RAGPipeline class orchestrating complete RAG workflow.

**Output:** src/rag_pipeline.py file created with RAGPipeline class

In [5]:
rag_pipeline_code = '''from typing import List, Dict, Tuple
from src.chunking import DocumentChunker
from src.prompts import RAGPrompts


class RAGPipeline:
    """RAG pipeline with hybrid retrieval and document tracking."""
    
    def __init__(self, bm25_retriever, semantic_retriever, llm, prompt_version="balanced"):
        """Initialize RAG pipeline with retrieval components and LLM."""
        self.bm25 = bm25_retriever
        self.semantic = semantic_retriever
        self.llm = llm
        self.corpus = None
        self.chunker = DocumentChunker(chunk_size=500, overlap=50)
        self.prompt_template = RAGPrompts.get_template(prompt_version)
    
    def retrieve_hybrid(self, query: str, top_k: int = 5) -> Tuple[List[int], List[str]]:
        """Hybrid retrieval: combine BM25 and semantic search."""
        bm25_results = self.bm25.search(query, top_k)
        semantic_results = self.semantic.search(query, top_k)
        
        bm25_ids = set(idx for idx, _ in bm25_results)
        semantic_ids = set(idx for idx, _ in semantic_results)
        combined_ids = list(bm25_ids | semantic_ids)[:top_k]
        
        documents = [self.corpus[idx] for idx in combined_ids]
        return combined_ids, documents
    
    def build_context(self, documents: List[str], max_tokens: int = 2000) -> str:
        """Build context from documents with token limit."""
        context = ""
        token_count = 0
        
        for i, doc in enumerate(documents, 1):
            review_text = "Review " + str(i) + ": " + doc
            token_count += len(review_text.split())
            
            if token_count > max_tokens:
                break
            
            context += review_text + " "
        
        return context
    
    def generate(self, query: str, context: str) -> str:
        """Generate answer using LLM."""
        prompt = self.prompt_template.format(context=context, question=query)
        
        try:
            return self.llm.invoke(prompt)
        except Exception as e:
            return "Error: " + str(e)
    
    def invoke(self, query: str, top_k: int = 5) -> Dict:
        """Complete RAG pipeline: retrieve -> context -> generate."""
        doc_ids, documents = self.retrieve_hybrid(query, top_k)
        context = self.build_context(documents)
        answer = self.generate(query, context)
        
        return {
            "query": query,
            "documents_retrieved": len(documents),
            "context_length": len(context),
            "answer": answer,
            "retrieved_doc_ids": doc_ids
        }
'''

rag_path = project_root / "src" / "rag_pipeline.py"
with open(rag_path, 'w') as f:
    f.write(rag_pipeline_code)

print(f"Created: src/rag_pipeline.py")
print(f"  Class: RAGPipeline")
print(f"  Methods: retrieve_hybrid, build_context, generate, invoke")

Created: src/rag_pipeline.py
  Class: RAGPipeline
  Methods: retrieve_hybrid, build_context, generate, invoke


## 6. Verify All Files Generated

Checks that all 3 modules were created successfully WITHOUT importing them (lightweight verification).

**Output:** File list - NO kernel crash (memory optimized for MacBook)

In [ ]:
import os

files_to_check = [
    (project_root / "src" / "chunking.py", "DocumentChunker"),
    (project_root / "src" / "prompts.py", "RAGPrompts"),
    (project_root / "src" / "rag_pipeline.py", "RAGPipeline")
]

print("Generated Files Summary")
print("=" * 60)

all_exist = True
for file_path, class_name in files_to_check:
    if file_path.exists():
        size = os.path.getsize(file_path)
        print(f"\nOK: {file_path.name}")
        print(f"  Class: {class_name}")
        print(f"  Size: {size} bytes")
    else:
        print(f"\nMISSING: {file_path.name}")
        all_exist = False

print()
print("=" * 60)
if all_exist:
    print("All RAG files generated successfully")
    print("\nReady to use with app.py")
else:
    print("Some files failed to generate")

Generated Files Summary

OK: chunking.py
  Class: DocumentChunker
  Size: 714 bytes

OK: prompts.py
  Class: RAGPrompts
  Size: 741 bytes

OK: rag_pipeline.py
  Class: RAGPipeline
  Size: 2480 bytes

✓ All RAG files generated successfully

Ready to use with app.py
